In [3]:
import cv2
import numpy as np
from ultralytics import YOLO

def main():
    # 1. Initialize YOLO (using the 'nano' model for maximum local efficiency)
    model = YOLO("yolov8n.pt")

    # 2. Open Video Stream
    # Replace with 0 for webcam, or an RTSP/HTTP stream URL
    cap = cv2.VideoCapture("videos/ref.mp4")

    # 3. Configure Classical CV Parameters
    # Shi-Tomasi corner detection parameters (The "Strong Descriptors")
    feature_params = dict(maxCorners=50,
                          qualityLevel=0.1,
                          minDistance=7,
                          blockSize=7)

    # Lucas-Kanade optical flow parameters
    lk_params = dict(winSize=(15, 15),
                     maxLevel=2,
                     criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

    # Pipeline State Variables
    detect_interval = 30  # Run YOLO every 30 frames
    threshold = 10
    frame_idx = 0
    tracked_objects = []  # Stores dicts: {'bbox': [x1,y1,x2,y2], 'points': np.array}
    prev_gray = None

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Convert to grayscale for classical CV operations
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        vis_frame = frame.copy()

        # ==========================================
        # PHASE 1: KEYFRAME (Deep Learning / YOLO)
        # ==========================================
        if frame_idx % detect_interval == 0 or not tracked_objects:
            tracked_objects = []

            # Run inference (verbose=False to keep console clean)
            results = model(frame, verbose=False)
            boxes = results[0].boxes.xyxy.cpu().numpy()

            for box in boxes:
                x1, y1, x2, y2 = map(int, box)

                # Create a blank mask and open a "window" for the bounding box
                mask = np.zeros_like(gray)
                mask[y1:y2, x1:x2] = 255

                # Extract strong features strictly inside the bounding box
                p0 = cv2.goodFeaturesToTrack(gray, mask=mask, **feature_params)

                # If features are found, initialize tracking for this object
                if p0 is not None:
                    tracked_objects.append({'bbox': [x1, y1, x2, y2], 'points': p0})

        # ==========================================
        # PHASE 2: P-FRAME (Classical / Lucas-Kanade)
        # ==========================================
        else:
            new_tracked_objects = []
            for obj in tracked_objects:
                p0 = obj['points']
                old_bbox = obj['bbox']

                # Calculate optical flow to find the new positions of the points
                p1, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, gray, p0, None, **lk_params)

                # Keep only points where tracking status == 1 (successfully tracked)
                good_new = p1[st == 1]
                good_old = p0[st == 1]

                # Safety Check: If the object loses too many tracking points, drop it.
                # It will be re-acquired on the next YOLO keyframe interval.
                if len(good_new) < threshold:
                    continue

                # Calculate the median movement of the points to avoid background snagging
                dx = np.median(good_new[:, 0] - good_old[:, 0])
                dy = np.median(good_new[:, 1] - good_old[:, 1])

                # Shift the bounding box coordinates by the median delta
                new_bbox = [
                    int(old_bbox[0] + dx),
                    int(old_bbox[1] + dy),
                    int(old_bbox[2] + dx),
                    int(old_bbox[3] + dy)
                ]

                # Reshape points for the next iteration's LK input requirements
                new_tracked_objects.append({
                    'bbox': new_bbox,
                    'points': good_new.reshape(-1, 1, 2)
                })

            tracked_objects = new_tracked_objects

        # ==========================================
        # VISUALIZATION & STATE UPDATE
        # ==========================================
        for obj in tracked_objects:
            x1, y1, x2, y2 = obj['bbox']

            # Draw the propagated bounding box
            cv2.rectangle(vis_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Draw the Lucas-Kanade tracking points
            for p in obj['points']:
                px, py = p.ravel()
                cv2.circle(vis_frame, (int(px), int(py)), 3, (0, 0, 255), -1)

        cv2.imshow("YOLO Keyframe + LK Tracking", vis_frame)

        # Store current frame for the next LK calculation
        prev_gray = gray.copy()
        frame_idx += 1

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()